# HWANGE — Colab driver

This notebook runs the pipeline; it does not contain it. All logic lives in `src/`.
Set the runtime to a **GPU** before running (Runtime > Change runtime type > T4).


## 1. Clone and install

In [ ]:
# Idempotent: re-running this must not clone inside the previous clone.
# A nested checkout (.../TRI-AI-hwange-proj/TRI-AI-hwange-proj) runs stale code from
# an unexpected working directory, which is confusing to diagnose later.
import os, subprocess

REPO_DIR = "/content/TRI-AI-hwange-proj"
REPO_URL = "https://github.com/semereherruy/TRI-AI-hwange-proj.git"

os.chdir("/content")
if os.path.isdir(os.path.join(REPO_DIR, ".git")):
    print("repository already present — updating")
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)

os.chdir(REPO_DIR)
print("working directory:", os.getcwd())

!pip install -q -r requirements.txt
!python check_env.py

### Fail-fast helper\n\nStops the notebook at the first failing step instead of cascading.

In [ ]:
# Shell cells with `!` do not raise when a command fails, and subprocess output
# goes to the kernel rather than the cell. This helper captures both streams,
# prints them, and stops the notebook at the actual failure.
import subprocess, sys

def run(command: str) -> None:
    print(f"$ {command}", flush=True)
    result = subprocess.run(
        command, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
    )
    print(result.stdout, flush=True)
    if result.returncode != 0:
        raise SystemExit(f"STOPPED (exit {result.returncode}): {command}")
    print("ok\n", flush=True)

## 2. Credentials

Gemma and AfriHate are gated. Add your token to the Colab **Secrets** panel (key icon)
as `HF_TOKEN` — never paste it into a cell.

In [ ]:
# Two separate things have to be true: the secret must reach this kernel, and the
# account behind it must have accepted each gated licence. Both are checked here,
# because the alternative is a 401 deep inside extraction long after the GPU is warm.
import os
from google.colab import userdata

try:
    token = userdata.get("HF_TOKEN")
except Exception as exc:
    raise SystemExit(
        "Could not read the HF_TOKEN secret.\n"
        "  Key icon in the left sidebar -> add a secret named exactly HF_TOKEN,\n"
        "  paste your token, and turn 'Notebook access' ON."
    ) from exc

os.environ["HF_TOKEN"] = token

# login() writes the token into the runtime's HuggingFace cache, so every
# subprocess started later picks it up regardless of environment inheritance.
from huggingface_hub import login, whoami, auth_check
login(token=token, add_to_git_credential=False)
print("authenticated as:", whoami()["name"])

for repo_id, repo_type in [("google/gemma-3-1b-pt", "model"), ("afrihate/afrihate", "dataset")]:
    try:
        auth_check(repo_id, repo_type=repo_type)
        print(f"access OK: {repo_id}")
    except Exception as exc:
        raise SystemExit(
            f"No access to {repo_id} ({type(exc).__name__}).\n"
            f"  Open https://huggingface.co/{repo_id} while signed in and accept the licence,\n"
            f"  then re-run this cell."
        )

## 3. Fetch the third-party datasets

The vendored repos are git-ignored, so HateXplain is re-fetched here. AfriHate and
ToxiGen come from the Hub via the loaders.

In [ ]:
run("mkdir -p data/raw")
run("git clone -q --depth 1 https://github.com/hate-alert/HateXplain.git data/raw/HateXplain-master || true")
run("ls data/raw/HateXplain-master/Data")

## 4. Data pipeline (Phases 2-7)

Each step writes a report. Edit `configs/data.yaml` to change a labelling policy;
every artefact records the policy that produced it.

In [ ]:
run("python -m src.data.inspection")      # Phase 2: inspection reports
run("python -m src.data.canonical")       # Phase 4: canonical dataset
run("python -m src.data.quality")         # Phase 5: quality + leakage report
run("python -m src.data.splits")          # Phase 6: group-level splits
run("python -m src.analysis.baselines")   # Phase 7: TF-IDF baselines

## 5. Gemma hidden-state extraction (Phase 8)

Smoke-test the path on a few rows first, then run the full extraction. The model is
frozen and used in inference mode only.

In [ ]:
# smoke-test the path on a few rows first, then the full extraction
run("python -m src.probing.extraction --splits train test --limit 32 --output /tmp/smoke")
run("python -m src.probing.extraction --splits train val test probe")

## 6. Layer-wise probes and controls (Phases 9-10)

In [ ]:
run("python -m src.probing.probes --embeddings data/embeddings")

## 7. Layer curve

The primary Phase 9 output: performance as a function of depth, read against the
Phase 7 TF-IDF baseline and the Phase 10 control floors.

In [ ]:
run("python -m src.probing.probes --embeddings data/embeddings")